In [0]:
%sql
-- 1. Create a project subfolder (schema)
CREATE SCHEMA IF NOT EXISTS db_retail_analytics.supply_chain_project;

-- 2. Create your "raw" container (volume) inside it
CREATE VOLUME IF NOT EXISTS db_retail_analytics.supply_chain_project.raw_files;


In [0]:
from pyspark.sql.functions import *

volume_path="/Volumes/db_retail_analytics/supply_chain_project/raw_files/"
source_dir=f"{volume_path}"
checkpoint_dir=f"{volume_path}checkpoints/"


# Injesting the raw orders in a readstream
orders_stream=(
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("header","true")
    #.option("cloudFiles.inferColumnTypes","true") because retries_exceed meaning it's taking too long ot infer schema so we are going to replace with bottom line and later we can use data type in transformations(silver layer)
    .option("cloudFiles.mergeSchema", "true") 
    .option("cloudFiles.schemaLocation",f"{checkpoint_dir}orders/schema")
    .load(f"{source_dir}olist_orders_dataset.csv")
)
orders_processed_wmetadata=(
    orders_stream
    .withColumn("_input_file_name",input_file_name())
    .withColumn("_processed_timestamp",current_timestamp())
)
query_orders=(
    orders_processed_wmetadata
    .writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint_dir}orders/checkpoint")
    .outputMode("append")
    .trigger(availableNow=True) # Triggering the stream to manually stop after the first batch(whole dataset)
    .toTable("db_retail_analytics.supply_chain_project.bronze_orders")
)

In [0]:
'''
Injestion for OLIST order_items
'''
order_items_stream=(
    spark
    .readStream
    .format("cloudFiles")
    .option("cloudFiles.format","csv")
    .option("header","true")
    .option("cloudFiles.inferColumnTypes","true")
    .option("cloudFiles.schemaLocation",f"{checkpoint_dir}order_items/schema")
    .load(f"{source_dir}olist_order_items_dataset.csv")
)

#This is the bronze data frame which has the metadata along with the data init

order_items_processed_wmetadata=(
    order_items_stream
    .withColumn("_input_file_name",input_file_name())
    .withColumn("_processed_timestamp",current_timestamp())
)

#writing in to delta table
query_order_items=(
    order_items_processed_wmetadata
    .writeStream
    .format("delta")
    .option("checkpointLocation",f"{checkpoint_dir}order_items/checkpoint")
    .outputMode("append")
    .trigger(availableNow=True) # Triggering the stream to manually stop after the first batch(whole dataset)
    .toTable("db_retail_analytics.supply_chain_project.bronze_order_items")
)

In [0]:
dbutils.fs.rm("/Volumes/db_retail_analytics/supply_chain_project/raw_files/checkpoints/orders", True)


In [0]:
from pyspark.sql.functions import current_timestamp, col  # Added 'col' here

# 1. Define paths precisely
volume_path = "/Volumes/db_retail_analytics/supply_chain_project/raw_files/"
source_file = f"{volume_path}olist_orders_dataset.csv"

# 2. Read as a stable, static batch 
orders_df = (spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true") 
    .load(source_file)
)

# 3. Add the exact mandatory tracking metadata using Unity Catalog's _metadata column
orders_bronze_df = (orders_df
    # Updated input_file_name() to col("_metadata.file_path")
    .withColumn("_input_file_name", col("_metadata.file_path"))
    .withColumn("_processed_timestamp", current_timestamp())
)

# 4. Write directly into the Unity Catalog Managed Table
(orders_bronze_df.write
    .format("delta")
    .mode("append")
    .saveAsTable("db_retail_analytics.supply_chain_project.bronze_orders")
)

print("Orders Table Ingested & Registered Successfully!")


In [0]:
%sql
select * from db_retail_analytics.supply_chain_project.bronze_orders

In [0]:
from pyspark.sql import functions as F
volume_path="/Volumes/db_retail_analytics/supply_chain_project/raw_files"

In [0]:
'''
Phase 1: injesting sellers data set in to bronze layer
'''

In [0]:
sellers_df=(
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(f"{volume_path}/olist_sellers_dataset.csv")
)
sellers_bronze_df=(
    sellers_df
    .withColumn("_input_file_name", F.col("_metadata.file_path"))
    .withColumn("_processed_timestamp",F.current_timestamp())
)
(
    sellers_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.bronze_sellers")
)

In [0]:
products_df=(
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(f"{volume_path}/olist_products_dataset.csv")
)
products_bronze_df=(
    products_df
    .withColumn("_input_file_name",F.col("_metadata.file_path"))
    .withColumn("_processed_timestamp",F.current_timestamp())
)
(
    products_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.bronze_products")
)

In [0]:
reviews_df=(
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(f"{volume_path}/olist_order_reviews_dataset.csv")
)
reviews_bronze_df=(
    reviews_df
    .withColumn("_input_file_name",F.col("_metadata.file_path"))
    .withColumn("_processed_timestamp",F.current_timestamp())
)
(
    reviews_bronze_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema","true")
    .saveAsTable("db_retail_analytics.supply_chain_project.bronze_reviews")
)

In [0]:
payments_df=(
    spark
    .read
    .format("csv")
    .option("header","true")
    .option("inferSchema","true")
    .load(f"{volume_path}/olist_order_payments_dataset.csv")
)
payments_bronze_df=(
    payments_df
    .withColumn("_input_file_name",F.col("_metadata.file_path"))
    .withColumn("_processed_time",F.current_timestamp())
)

(
    payments_bronze_df
    .write
    .format("delta")
    .option("overwriteSchema","true")
    .mode("overwrite")
    .saveAsTable("db_retail_analytics.supply_chain_project.bronze_payments")
)